### Installation

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

### Nusantara-1.8b

In [ ]:
import time
import torch
import math
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# --- Konfigurasi Model dan Tokenizer ---
# Ganti dengan model Anda
model_name = "kalisai/Nusantara-1.8b-Indo-Chat" # Contoh: "unsloth/Llama-3.2-1B-Instruct" atau "kalisai/Nusantara-1.8b-Indo-Chat"
max_seq_length = 8192
load_in_4bit = True
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Memuat model {model_name} ke {device}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit
)
print("Model berhasil dimuat.")

FastLanguageModel.for_inference(model)
# --- Prompt untuk Pengujian ---
prompt_text = "Jelaskan menara tertinggi di dunia."
messages = [
    {"role": "system", "content": "Kamu adalah asisten AI yang pintar."},
    {"role": "user", "content": prompt_text}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False, # Penting agar hasilnya berupa string yang bisa di-tokenize ulang
    add_generation_prompt=True
)
print(f"\nPrompt yang diformat:\n---\n{text}\n---")

# Tokenisasi input
model_inputs = tokenizer([text], return_tensors="pt").to(device)

# --- Parameter Generasi ---
max_new_tokens_to_generate = 512 # Jumlah maksimum token baru yang akan dihasilkan

print("\nMemulai pengujian performa...")
# --- 1. Mengukur TTFT (Time To First Token) ---
start_time_ttft = time.time()
with torch.no_grad():
    # Hasilkan hanya 1 token untuk mengukur TTFT
    # Pastikan output_scores=True untuk kompatibilitas dengan Perplexity di kemudian hari
    _ = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True # Penting
    )
end_time_ttft = time.time()
ttft = end_time_ttft - start_time_ttft
print(f"**Time To First Token (TTFT): {ttft:.4f} seconds**")

# --- 2. Mengukur Latensi Total, Throughput, dan Perplexity ---
start_time_total = time.time() # Mulai waktu untuk Latensi Total

with torch.no_grad():
    generated_outputs = model.generate(
        model_inputs.input_ids,
        max_new_tokens=max_new_tokens_to_generate,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True # Sangat penting untuk menghitung perplexity
    )
end_time_total = time.time()
total_latency = end_time_total - start_time_total

# Ekstrak token yang dihasilkan (tanpa prompt input)
generated_token_ids = generated_outputs.sequences[:, model_inputs.input_ids.shape[1]:]
num_generated_tokens = generated_token_ids.shape[1]

# Menghitung Throughput (Tokens per Second)
if num_generated_tokens > 0:
    throughput = num_generated_tokens / total_latency
else:
    throughput = 0.0 # Tidak ada token yang dihasilkan
print(f"**Total Latency: {total_latency:.4f} seconds**")
print(f"**Tokens Generated: {num_generated_tokens}**")
print(f"**Throughput (Tokens/second): {throughput:.2f} TPS**")

# --- Menghitung Perplexity ---
total_neg_log_likelihood = 0.0

if generated_outputs.scores: # Pastikan ada scores yang dihasilkan
    logits = torch.stack(generated_outputs.scores, dim=1) # Shape: (batch_size, num_generated_tokens, vocab_size)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Asumsi batch_size = 1
    for j in range(num_generated_tokens):
        token_id = generated_token_ids[0, j].item() # Ambil ID token yang dihasilkan

        # Abaikan special token (seperti EOS dan PAD) untuk perhitungan perplexity
        if token_id not in [tokenizer.eos_token_id, tokenizer.pad_token_id]:
            total_neg_log_likelihood += -log_probs[0, j, token_id].item()

    if num_generated_tokens > 0:
        average_neg_log_likelihood = total_neg_log_likelihood / num_generated_tokens
        perplexity = math.exp(average_neg_log_likelihood)
        print(f"**Perplexity dari output yang dihasilkan: {perplexity:.4f}**")
    else:
        print("Tidak ada token yang relevan yang dihasilkan untuk menghitung perplexity.")
else:
    print("Scores tidak tersedia untuk menghitung perplexity. Pastikan 'output_scores=True' diaktifkan.")

# --- Decode dan Cetak Respons ---
response = tokenizer.batch_decode(generated_token_ids, skip_special_tokens=True)[0]
print("\n--- Respons Model ---")
print(response)

Memuat model kalisai/Nusantara-1.8b-Indo-Chat ke cuda...
==((====))==  Unsloth 2025.7.3: Fast Qwen2 patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

kalisai/Nusantara-1.8b-Indo-Chat does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model berhasil dimuat.

Prompt yang diformat:
---
<|im_start|>system
Kamu adalah asisten AI yang pintar.<|im_end|>
<|im_start|>user
Jelaskan menara tertinggi di dunia.<|im_end|>
<|im_start|>assistant

---

Memulai pengujian performa...
**Time To First Token (TTFT): 0.0939 seconds**
**Total Latency: 12.8387 seconds**
**Tokens Generated: 383**
**Throughput (Tokens/second): 29.83 TPS**
**Perplexity dari output yang dihasilkan: 1.5457**

--- Respons Model ---
Sebagai asisten AI yang pintar, saya akan memberikan informasi tentang menara tertinggi di dunia. 

Menara tertinggi di dunia adalah Menara Burj Khalifa di Dubai, Uni Emirat Arab. Menara ini memiliki ketinggian 828 meter dan merupakan salah satu bangunan tertinggi di dunia. Menara ini dibangun pada tahun 2009 dan selesai pada tahun 2010. Menara ini memiliki 160 lantai dan memiliki 100 lift untuk mengangkut penumpang dari lantai ke lanta

### Qwen3-0.6B

In [ ]:
import time
import torch
import math
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# --- Konfigurasi Model dan Tokenizer ---
# Menggunakan model Qwen/Qwen2-0.5B-Instruct atau unsloth/Qwen2-0.5B-Instruct-bnb-4bit
# Perlu diperhatikan bahwa model Unsloth cenderung lebih dioptimalkan.
model_name = "unsloth/Qwen3-0.6B" # Menggunakan versi Unsloth yang dioptimalkan
max_seq_length = 8192
load_in_4bit = True
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Memuat model {model_name} ke {device}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit
)
print("Model berhasil dimuat.")

# Pastikan model diatur untuk inferensi
FastLanguageModel.for_inference(model)

# --- Penyesuaian Chat Template untuk Qwen 2 ---
# Qwen 2 (dan umumnya Qwen) menggunakan format ChatML
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml", # Menggunakan chatml template
    # mapping tidak diperlukan jika menggunakan template standar seperti chatml
)
print("Chat template disesuaikan untuk ChatML (Qwen 2).")

# --- Prompt untuk Pengujian ---
prompt_text = "Jelaskan menara tertinggi di dunia."
messages = [
    {"role": "system", "content": "Kamu adalah asisten AI yang pintar."},
    {"role": "user", "content": prompt_text}
]

# Konversi messages ke format input model menggunakan chat template yang sudah disesuaikan
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"\nPrompt yang diformat:\n---\n{text}\n---")

# Tokenisasi input
model_inputs = tokenizer([text], return_tensors="pt").to(device)

# --- Parameter Generasi ---
max_new_tokens_to_generate = 512 # Jumlah maksimum token baru yang akan dihasilkan

print("\nMemulai pengujian performa...")

### Time To First Token (TTFT)
start_time_ttft = time.time()
with torch.no_grad():
    # Hasilkan hanya 1 token untuk mengukur TTFT
    _ = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True
    )
end_time_ttft = time.time()
ttft = end_time_ttft - start_time_ttft
print(f"**Time To First Token (TTFT): {ttft:.4f} seconds**")


### Latensi Total, Throughput, dan Perplexity
start_time_total = time.time() # Mulai waktu untuk Latensi Total

with torch.no_grad():
    generated_outputs = model.generate(
        model_inputs.input_ids,
        max_new_tokens=max_new_tokens_to_generate,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True # Sangat penting untuk menghitung perplexity
    )
end_time_total = time.time()
total_latency = end_time_total - start_time_total

# Ekstrak token yang dihasilkan (tanpa prompt input)
generated_token_ids = generated_outputs.sequences[:, model_inputs.input_ids.shape[1]:]
num_generated_tokens = generated_token_ids.shape[1]

# Menghitung Throughput (Tokens per Second)
if num_generated_tokens > 0:
    throughput = num_generated_tokens / total_latency
else:
    throughput = 0.0
print(f"**Total Latency: {total_latency:.4f} seconds**")
print(f"**Tokens Generated: {num_generated_tokens}**")
print(f"**Throughput (Tokens/second): {throughput:.2f} TPS**")

# --- Menghitung Perplexity ---
total_neg_log_likelihood = 0.0

if generated_outputs.scores:
    logits = torch.stack(generated_outputs.scores, dim=1) # Shape: (batch_size, num_generated_tokens, vocab_size)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Asumsi batch_size = 1
    for j in range(num_generated_tokens):
        token_id = generated_token_ids[0, j].item()

        # Abaikan special token (seperti EOS dan PAD) untuk perhitungan perplexity
        if token_id not in [tokenizer.eos_token_id, tokenizer.pad_token_id]:
            total_neg_log_likelihood += -log_probs[0, j, token_id].item()

    if num_generated_tokens > 0:
        average_neg_log_likelihood = total_neg_log_likelihood / num_generated_tokens
        perplexity = math.exp(average_neg_log_likelihood)
        print(f"**Perplexity dari output yang dihasilkan: {perplexity:.4f}**")
    else:
        print("Tidak ada token yang relevan yang dihasilkan untuk menghitung perplexity.")
else:
    print("Scores tidak tersedia untuk menghitung perplexity. Pastikan 'output_scores=True' diaktifkan.")

### Respons Model
response = tokenizer.batch_decode(generated_token_ids, skip_special_tokens=True)[0]
print(response)

Memuat model unsloth/Qwen3-0.6B ke cuda...
==((====))==  Unsloth 2025.7.3: Fast Qwen3 patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


Model berhasil dimuat.
Chat template disesuaikan untuk ChatML (Qwen 2).

Prompt yang diformat:
---
<|im_start|>system
Kamu adalah asisten AI yang pintar.<|im_end|>
<|im_start|>user
Jelaskan menara tertinggi di dunia.<|im_end|>
<|im_start|>assistant

---

Memulai pengujian performa...
**Time To First Token (TTFT): 0.1112 seconds**
**Total Latency: 10.9340 seconds**
**Tokens Generated: 234**
**Throughput (Tokens/second): 21.40 TPS**
**Perplexity dari output yang dihasilkan: 1.4925**
<think>
Okay, the user is asking for the highest mountain in the world. I need to provide a clear and concise answer. First, I should recall the tallest mountain in the world. The tallest mountain is Mount Everest, located in the Himalayas. I should mention its elevation, the height, and the location. It's important to explain why it's the highest, like the geological features. I should also include the year and the number of climbers. Need to make sure the information is accurate and up-to-date. Let me check

### Qwen3-1.7B

In [ ]:
import time
import torch
import math
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# --- Konfigurasi Model dan Tokenizer ---
# Menggunakan model Qwen/Qwen2-0.5B-Instruct atau unsloth/Qwen2-0.5B-Instruct-bnb-4bit
# Perlu diperhatikan bahwa model Unsloth cenderung lebih dioptimalkan.
model_name = "unsloth/Qwen3-1.7B" # Menggunakan versi Unsloth yang dioptimalkan
max_seq_length = 8192
load_in_4bit = True
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Memuat model {model_name} ke {device}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit
)
print("Model berhasil dimuat.")

# Pastikan model diatur untuk inferensi
FastLanguageModel.for_inference(model)

# --- Penyesuaian Chat Template untuk Qwen 2 ---
# Qwen 2 (dan umumnya Qwen) menggunakan format ChatML
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml", # Menggunakan chatml template
    # mapping tidak diperlukan jika menggunakan template standar seperti chatml
)
print("Chat template disesuaikan untuk ChatML (Qwen 2).")

# --- Prompt untuk Pengujian ---
prompt_text = "Jelaskan menara tertinggi di dunia."
messages = [
    {"role": "system", "content": "Kamu adalah asisten AI yang pintar."},
    {"role": "user", "content": prompt_text}
]

# Konversi messages ke format input model menggunakan chat template yang sudah disesuaikan
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"\nPrompt yang diformat:\n---\n{text}\n---")

# Tokenisasi input
model_inputs = tokenizer([text], return_tensors="pt").to(device)

# --- Parameter Generasi ---
max_new_tokens_to_generate = 512 # Jumlah maksimum token baru yang akan dihasilkan

print("\nMemulai pengujian performa...")

### Time To First Token (TTFT)
start_time_ttft = time.time()
with torch.no_grad():
    # Hasilkan hanya 1 token untuk mengukur TTFT
    _ = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True
    )
end_time_ttft = time.time()
ttft = end_time_ttft - start_time_ttft
print(f"**Time To First Token (TTFT): {ttft:.4f} seconds**")


### Latensi Total, Throughput, dan Perplexity
start_time_total = time.time() # Mulai waktu untuk Latensi Total

with torch.no_grad():
    generated_outputs = model.generate(
        model_inputs.input_ids,
        max_new_tokens=max_new_tokens_to_generate,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True # Sangat penting untuk menghitung perplexity
    )
end_time_total = time.time()
total_latency = end_time_total - start_time_total

# Ekstrak token yang dihasilkan (tanpa prompt input)
generated_token_ids = generated_outputs.sequences[:, model_inputs.input_ids.shape[1]:]
num_generated_tokens = generated_token_ids.shape[1]

# Menghitung Throughput (Tokens per Second)
if num_generated_tokens > 0:
    throughput = num_generated_tokens / total_latency
else:
    throughput = 0.0
print(f"**Total Latency: {total_latency:.4f} seconds**")
print(f"**Tokens Generated: {num_generated_tokens}**")
print(f"**Throughput (Tokens/second): {throughput:.2f} TPS**")

# --- Menghitung Perplexity ---
total_neg_log_likelihood = 0.0

if generated_outputs.scores:
    logits = torch.stack(generated_outputs.scores, dim=1) # Shape: (batch_size, num_generated_tokens, vocab_size)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Asumsi batch_size = 1
    for j in range(num_generated_tokens):
        token_id = generated_token_ids[0, j].item()

        # Abaikan special token (seperti EOS dan PAD) untuk perhitungan perplexity
        if token_id not in [tokenizer.eos_token_id, tokenizer.pad_token_id]:
            total_neg_log_likelihood += -log_probs[0, j, token_id].item()

    if num_generated_tokens > 0:
        average_neg_log_likelihood = total_neg_log_likelihood / num_generated_tokens
        perplexity = math.exp(average_neg_log_likelihood)
        print(f"**Perplexity dari output yang dihasilkan: {perplexity:.4f}**")
    else:
        print("Tidak ada token yang relevan yang dihasilkan untuk menghitung perplexity.")
else:
    print("Scores tidak tersedia untuk menghitung perplexity. Pastikan 'output_scores=True' diaktifkan.")

### Respons Model
response = tokenizer.batch_decode(generated_token_ids, skip_special_tokens=True)[0]
print(response)

Memuat model unsloth/Qwen3-1.7B ke cuda...
==((====))==  Unsloth 2025.7.3: Fast Qwen3 patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.41G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Model berhasil dimuat.
Chat template disesuaikan untuk ChatML (Qwen 2).

Prompt yang diformat:
---
<|im_start|>system
Kamu adalah asisten AI yang pintar.<|im_end|>
<|im_start|>user
Jelaskan menara tertinggi di dunia.<|im_end|>
<|im_start|>assistant

---

Memulai pengujian performa...
**Time To First Token (TTFT): 0.1105 seconds**
**Total Latency: 23.5289 seconds**
**Tokens Generated: 512**
**Throughput (Tokens/second): 21.76 TPS**
**Perplexity dari output yang dihasilkan: 1.2872**
<think>
Okay, the user is asking about the tallest tower in the world. Let me start by recalling the main contenders. The Burj Khalifa in Dubai comes to mind. I remember it was built to be the tallest building, but I need to check the exact height. I think it's over 828 meters. Then there's the Shanghai Tower in China, which is a contender but hasn't reached the same height. The CN Tower in Toronto is another one, but it's not as tall as the Burj Khalifa. Also, the Petronas Tower in Kuala Lumpur was the talle

### Gemma 3-1b-it

In [ ]:
import time
import torch
import math
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# --- Konfigurasi Model dan Tokenizer ---
model_name = "unsloth/gemma-3-1b-it" # Model Gemma 3 1B Instruct
max_seq_length = 8192
load_in_4bit = True
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model {model_name} to {device}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit
)
print("Model loaded successfully.")

# Ensure the model is set for inference
FastLanguageModel.for_inference(model)

# --- Adjusting Chat Template for Gemma ---
# Gemma models often use a specific template. Unsloth provides "gemma" or "gemma_chatml".
# Let's try "gemma" first. If the output format is off, you might try "gemma_chatml".
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma", # Using the specific Gemma chat template
    # mapping is not typically needed for standard Gemma templates
)
print("Chat template adjusted for Gemma.")

# --- Prompt for Testing ---
prompt_text = "Jelaskan menara tertinggi di dunia."
messages = [
    {"role": "system", "content": "Kamu adalah asisten AI yang pintar."},
    {"role": "user", "content": prompt_text}
]

# Convert messages to model input format using the adjusted chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"\nFormatted Prompt:\n---\n{text}\n---")

# Tokenize input
model_inputs = tokenizer([text], return_tensors="pt").to(device)

# --- Generation Parameters ---
max_new_tokens_to_generate = 512 # Maximum number of new tokens to generate

print("\nStarting performance evaluation...")
### Time To First Token (TTFT)
start_time_ttft = time.time()
with torch.no_grad():
    # Generate only 1 token to measure TTFT
    _ = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True
    )
end_time_ttft = time.time()
ttft = end_time_ttft - start_time_ttft
print(f"**Time To First Token (TTFT): {ttft:.4f} seconds**")

### Total Latency, Throughput, and Perplexity
start_time_total = time.time() # Start time for Total Latency

with torch.no_grad():
    generated_outputs = model.generate(
        model_inputs.input_ids,
        max_new_tokens=max_new_tokens_to_generate,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True # Crucial for calculating perplexity
    )
end_time_total = time.time()
total_latency = end_time_total - start_time_total

# Extract generated tokens (without the input prompt)
generated_token_ids = generated_outputs.sequences[:, model_inputs.input_ids.shape[1]:]
num_generated_tokens = generated_token_ids.shape[1]

# Calculate Throughput (Tokens per Second)
if num_generated_tokens > 0:
    throughput = num_generated_tokens / total_latency
else:
    throughput = 0.0
print(f"**Total Latency: {total_latency:.4f} seconds**")
print(f"**Tokens Generated: {num_generated_tokens}**")
print(f"**Throughput (Tokens/second): {throughput:.2f} TPS**")

# --- Calculate Perplexity ---
total_neg_log_likelihood = 0.0

if generated_outputs.scores:
    logits = torch.stack(generated_outputs.scores, dim=1) # Shape: (batch_size, num_generated_tokens, vocab_size)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Assuming batch_size = 1
    for j in range(num_generated_tokens):
        token_id = generated_token_ids[0, j].item()

        # Exclude special tokens (like EOS and PAD) from perplexity calculation
        if token_id not in [tokenizer.eos_token_id, tokenizer.pad_token_id]:
            total_neg_log_likelihood += -log_probs[0, j, token_id].item()

    if num_generated_tokens > 0:
        average_neg_log_likelihood = total_neg_log_likelihood / num_generated_tokens
        perplexity = math.exp(average_neg_log_likelihood)
        print(f"**Perplexity of generated output: {perplexity:.4f}**")
    else:
        print("No relevant tokens generated to calculate perplexity.")
else:
    print("Scores not available to calculate perplexity. Ensure 'output_scores=True' is enabled.")

### Model Response
response = tokenizer.batch_decode(generated_token_ids, skip_special_tokens=True)[0]
print(response)

Loading model unsloth/gemma-3-1b-it to cuda...
==((====))==  Unsloth 2025.7.3: Fast Gemma3 patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded successfully.
Chat template adjusted for Gemma.

Formatted Prompt:
---
<bos><start_of_turn>user
Kamu adalah asisten AI yang pintar. Jelaskan menara tertinggi di dunia.<end_of_turn>
<start_of_turn>model

---

Starting performance evaluation...
**Time To First Token (TTFT): 0.1810 seconds**
**Total Latency: 58.8699 seconds**
**Tokens Generated: 512**
**Throughput (Tokens/second): 8.70 TPS**
**Perplexity of generated output: 2.1759**
Baik, dengan senang hati saya akan m

### Llama-3.2-1B

In [ ]:
import time
import torch
import math
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# --- Konfigurasi Model dan Tokenizer ---
model_name = "unsloth/Llama-3.2-1B-Instruct" # Model Llama 3.2 1B Instruct
max_seq_length = 8192
load_in_4bit = True
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model {model_name} to {device}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit
)
print("Model loaded successfully.")

# Ensure the model is set for inference
FastLanguageModel.for_inference(model)

# --- Adjusting Chat Template for Llama 3.x ---
# Llama 3 and its variants (3.1, 3.2) use a specific chat template.
# Unsloth provides "llama-3.1" which is suitable.
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1", # Using the specific Llama 3.1 chat template
    # Mapping can be useful if your message format is different from standard
    # {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
)
print("Chat template adjusted for Llama 3.1.")

# --- Prompt for Testing ---
prompt_text = "Jelaskan menara tertinggi di dunia."
messages = [
    {"role": "system", "content": "Kamu adalah asisten AI yang pintar."},
    {"role": "user", "content": prompt_text}
]

# Convert messages to model input format using the adjusted chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"\nFormatted Prompt:\n---\n{text}\n---")

# Tokenize input
model_inputs = tokenizer([text], return_tensors="pt").to(device)

# --- Generation Parameters ---
max_new_tokens_to_generate = 512 # Maximum number of new tokens to generate

print("\nStarting performance evaluation...")

### Time To First Token (TTFT)
start_time_ttft = time.time()
with torch.no_grad():
    # Generate only 1 token to measure TTFT
    _ = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True
    )
end_time_ttft = time.time()
ttft = end_time_ttft - start_time_ttft
print(f"**Time To First Token (TTFT): {ttft:.4f} seconds**")

### Total Latency, Throughput, and Perplexity
start_time_total = time.time() # Start time for Total Latency

with torch.no_grad():
    generated_outputs = model.generate(
        model_inputs.input_ids,
        max_new_tokens=max_new_tokens_to_generate,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True # Crucial for calculating perplexity
    )
end_time_total = time.time()
total_latency = end_time_total - start_time_total

# Extract generated tokens (without the input prompt)
generated_token_ids = generated_outputs.sequences[:, model_inputs.input_ids.shape[1]:]
num_generated_tokens = generated_token_ids.shape[1]

# Calculate Throughput (Tokens per Second)
if num_generated_tokens > 0:
    throughput = num_generated_tokens / total_latency
else:
    throughput = 0.0
print(f"**Total Latency: {total_latency:.4f} seconds**")
print(f"**Tokens Generated: {num_generated_tokens}**")
print(f"**Throughput (Tokens/second): {throughput:.2f} TPS**")

# --- Calculate Perplexity ---
total_neg_log_likelihood = 0.0

if generated_outputs.scores:
    logits = torch.stack(generated_outputs.scores, dim=1) # Shape: (batch_size, num_generated_tokens, vocab_size)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Assuming batch_size = 1
    for j in range(num_generated_tokens):
        token_id = generated_token_ids[0, j].item()

        # Exclude special tokens (like EOS and PAD) from perplexity calculation
        if token_id not in [tokenizer.eos_token_id, tokenizer.pad_token_id]:
            total_neg_log_likelihood += -log_probs[0, j, token_id].item()

    if num_generated_tokens > 0:
        average_neg_log_likelihood = total_neg_log_likelihood / num_generated_tokens
        perplexity = math.exp(average_neg_log_likelihood)
        print(f"**Perplexity of generated output: {perplexity:.4f}**")
    else:
        print("No relevant tokens generated to calculate perplexity.")
else:
    print("Scores not available to calculate perplexity. Ensure 'output_scores=True' is enabled.")

### Model Response
response = tokenizer.batch_decode(generated_token_ids, skip_special_tokens=True)[0]
print(response)

Loading model unsloth/Llama-3.2-1B-Instruct to cuda...
==((====))==  Unsloth 2025.7.3: Fast Llama patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded successfully.
Chat template adjusted for Llama 3.1.

Formatted Prompt:
---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Kamu adalah asisten AI yang pintar.<|eot_id|><|start_header_id|>user<|end_header_id|>

Jelaskan menara tertinggi di dunia.<|eot_id|><|start_header_id|>assistant<|end_header_id|>


---

Starting performance evaluation...
**Time To First Token (TTFT): 0.0650 seconds

In [ ]:
import time
import torch
import math
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# --- Konfigurasi Model dan Tokenizer ---
model_name = "unsloth/Llama-3.2-1B-Instruct" # Model Llama 3.2 1B Instruct
max_seq_length = 8192
load_in_4bit = True
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model {model_name} to {device}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit
)
print("Model loaded successfully.")

# Ensure the model is set for inference
FastLanguageModel.for_inference(model)

# --- Adjusting Chat Template for Llama 3.x ---
# Llama 3 and its variants (3.1, 3.2) use a specific chat template.
# Unsloth provides "llama-3.1" which is suitable.
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1", # Using the specific Llama 3.1 chat template
    # Mapping can be useful if your message format is different from standard
    # {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
)
print("Chat template adjusted for Llama 3.1.")

# --- Prompt for Testing ---
prompt_text = "Nafion ditemukan pada akhir tahun berapa dan oleh siapa"
messages = [
    {"role": "system", "content": "Kamu adalah asisten AI yang pintar."},
    {"role": "user", "content": prompt_text}
]

# Convert messages to model input format using the adjusted chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"\nFormatted Prompt:\n---\n{text}\n---")

# Tokenize input
model_inputs = tokenizer([text], return_tensors="pt").to(device)

# --- Generation Parameters ---
max_new_tokens_to_generate = 512 # Maximum number of new tokens to generate

print("\nStarting performance evaluation...")

### Time To First Token (TTFT)
start_time_ttft = time.time()
with torch.no_grad():
    # Generate only 1 token to measure TTFT
    _ = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True
    )
end_time_ttft = time.time()
ttft = end_time_ttft - start_time_ttft
print(f"**Time To First Token (TTFT): {ttft:.4f} seconds**")

### Total Latency, Throughput, and Perplexity
start_time_total = time.time() # Start time for Total Latency

with torch.no_grad():
    generated_outputs = model.generate(
        model_inputs.input_ids,
        max_new_tokens=max_new_tokens_to_generate,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True # Crucial for calculating perplexity
    )
end_time_total = time.time()
total_latency = end_time_total - start_time_total

# Extract generated tokens (without the input prompt)
generated_token_ids = generated_outputs.sequences[:, model_inputs.input_ids.shape[1]:]
num_generated_tokens = generated_token_ids.shape[1]

# Calculate Throughput (Tokens per Second)
if num_generated_tokens > 0:
    throughput = num_generated_tokens / total_latency
else:
    throughput = 0.0
print(f"**Total Latency: {total_latency:.4f} seconds**")
print(f"**Tokens Generated: {num_generated_tokens}**")
print(f"**Throughput (Tokens/second): {throughput:.2f} TPS**")

# --- Calculate Perplexity ---
total_neg_log_likelihood = 0.0

if generated_outputs.scores:
    logits = torch.stack(generated_outputs.scores, dim=1) # Shape: (batch_size, num_generated_tokens, vocab_size)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Assuming batch_size = 1
    for j in range(num_generated_tokens):
        token_id = generated_token_ids[0, j].item()

        # Exclude special tokens (like EOS and PAD) from perplexity calculation
        if token_id not in [tokenizer.eos_token_id, tokenizer.pad_token_id]:
            total_neg_log_likelihood += -log_probs[0, j, token_id].item()

    if num_generated_tokens > 0:
        average_neg_log_likelihood = total_neg_log_likelihood / num_generated_tokens
        perplexity = math.exp(average_neg_log_likelihood)
        print(f"**Perplexity of generated output: {perplexity:.4f}**")
    else:
        print("No relevant tokens generated to calculate perplexity.")
else:
    print("Scores not available to calculate perplexity. Ensure 'output_scores=True' is enabled.")

### Model Response
response = tokenizer.batch_decode(generated_token_ids, skip_special_tokens=True)[0]
print(response)

Loading model unsloth/Llama-3.2-1B-Instruct to cuda...
==((====))==  Unsloth 2025.7.3: Fast Llama patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded successfully.
Chat template adjusted for Llama 3.1.

Formatted Prompt:
---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Kamu adalah asisten AI yang pintar.<|eot_id|><|start_header_id|>user<|end_header_id|>

Nafion ditemukan pada akhir tahun berapa dan oleh siapa<|eot_id|><|start_header_id|>assistant<|end_header_id|>


---

Starting performance evaluation...
**Time To First Token (T

### Llama-3.2-3B

In [ ]:
import time
import torch
import math
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# --- Konfigurasi Model dan Tokenizer ---
model_name = "unsloth/Llama-3.2-3B-Instruct" # Model Llama 3.2 1B Instruct
max_seq_length = 8192
load_in_4bit = True
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model {model_name} to {device}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit
)
print("Model loaded successfully.")

# Ensure the model is set for inference
FastLanguageModel.for_inference(model)

# --- Adjusting Chat Template for Llama 3.x ---
# Llama 3 and its variants (3.1, 3.2) use a specific chat template.
# Unsloth provides "llama-3.1" which is suitable.
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1", # Using the specific Llama 3.1 chat template
    # Mapping can be useful if your message format is different from standard
    # {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
)
print("Chat template adjusted for Llama 3.1.")

# --- Prompt for Testing ---
prompt_text = "Jelaskan menara tertinggi di dunia."
messages = [
    {"role": "system", "content": "Kamu adalah asisten AI yang pintar."},
    {"role": "user", "content": prompt_text}
]

# Convert messages to model input format using the adjusted chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"\nFormatted Prompt:\n---\n{text}\n---")

# Tokenize input
model_inputs = tokenizer([text], return_tensors="pt").to(device)

# --- Generation Parameters ---
max_new_tokens_to_generate = 512 # Maximum number of new tokens to generate

print("\nStarting performance evaluation...")

### Time To First Token (TTFT)
start_time_ttft = time.time()
with torch.no_grad():
    # Generate only 1 token to measure TTFT
    _ = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True
    )
end_time_ttft = time.time()
ttft = end_time_ttft - start_time_ttft
print(f"**Time To First Token (TTFT): {ttft:.4f} seconds**")

### Total Latency, Throughput, and Perplexity
start_time_total = time.time() # Start time for Total Latency

with torch.no_grad():
    generated_outputs = model.generate(
        model_inputs.input_ids,
        max_new_tokens=max_new_tokens_to_generate,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True # Crucial for calculating perplexity
    )
end_time_total = time.time()
total_latency = end_time_total - start_time_total

# Extract generated tokens (without the input prompt)
generated_token_ids = generated_outputs.sequences[:, model_inputs.input_ids.shape[1]:]
num_generated_tokens = generated_token_ids.shape[1]

# Calculate Throughput (Tokens per Second)
if num_generated_tokens > 0:
    throughput = num_generated_tokens / total_latency
else:
    throughput = 0.0
print(f"**Total Latency: {total_latency:.4f} seconds**")
print(f"**Tokens Generated: {num_generated_tokens}**")
print(f"**Throughput (Tokens/second): {throughput:.2f} TPS**")

# --- Calculate Perplexity ---
total_neg_log_likelihood = 0.0

if generated_outputs.scores:
    logits = torch.stack(generated_outputs.scores, dim=1) # Shape: (batch_size, num_generated_tokens, vocab_size)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Assuming batch_size = 1
    for j in range(num_generated_tokens):
        token_id = generated_token_ids[0, j].item()

        # Exclude special tokens (like EOS and PAD) from perplexity calculation
        if token_id not in [tokenizer.eos_token_id, tokenizer.pad_token_id]:
            total_neg_log_likelihood += -log_probs[0, j, token_id].item()

    if num_generated_tokens > 0:
        average_neg_log_likelihood = total_neg_log_likelihood / num_generated_tokens
        perplexity = math.exp(average_neg_log_likelihood)
        print(f"**Perplexity of generated output: {perplexity:.4f}**")
    else:
        print("No relevant tokens generated to calculate perplexity.")
else:
    print("Scores not available to calculate perplexity. Ensure 'output_scores=True' is enabled.")

### Model Response
response = tokenizer.batch_decode(generated_token_ids, skip_special_tokens=True)[0]
print(response)

Loading model unsloth/Llama-3.2-3B-Instruct to cuda...
==((====))==  Unsloth 2025.7.3: Fast Llama patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model loaded successfully.
Chat template adjusted for Llama 3.1.

Formatted Prompt:
---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Kamu adalah asisten AI yang pintar.<|eot_id|><|start_header_id|>user<|end_header_id|>

Jelaskan menara tertinggi di dunia.<|eot_id|><|start_header_id|>assistant<|end_header_id|>


---

Starting performance evaluation...
**Time To First Token (TTFT): 3.9978 seconds**
**Total Latency: 10.0218 seconds**
**Tokens Generated: 266**
**Throughput (Tokens/second): 26.54 TPS**
**Perplexity of generated output: 1.1627**
Tentu, saya dapat menjelaskan tentang menara tertinggi di dunia.

Menara tertinggi di dunia saat ini adalah Burj Khalifa, terletak di Dubai, Uni Emirat Arab. Berikut beberapa informasi menarik tentang Burj Khalifa:

1. **Tinggi**: Burj Khalifa memiliki tinggi sebesar 828 meter (2.722 kaki) di atas tanah.
2. **Desain**: Burj Khalifa dirancang oleh perusahaan arsitektur Ski

### Phi-3.5-mini

In [ ]:
import time
import torch
import math
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# --- Konfigurasi Model dan Tokenizer ---
model_name = "unsloth/Phi-3.5-mini-instruct" # Model Phi-3.5-mini-instruct
max_seq_length = 8192 # Sesuaikan dengan kebutuhan memori GPU Anda
load_in_4bit = True # Memuat dalam 4-bit untuk efisiensi memori
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Memuat model {model_name} ke {device}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit
)
print("Model berhasil dimuat.")

# Pastikan model diatur untuk inferensi
FastLanguageModel.for_inference(model)

# --- Penyesuaian Chat Template untuk Phi-3.5 ---
# Model Phi-3.5 memiliki template chat spesifik. Unsloth menyediakan "phi-35".
tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-35", # Menggunakan chat template khusus Phi-3.5
    # mapping tidak diperlukan jika menggunakan template standar Phi-3.5
)
print("Chat template disesuaikan untuk Phi-3.5.")

# --- Prompt untuk Pengujian ---
# Gunakan salah satu dari pertanyaan umum yang disepakati untuk konsistensi
prompt_text = "Jelaskan menara tertinggi di dunia."
# prompt_text = "Apa perbedaan antara iklim tropis dan iklim subtropis?"
# prompt_text = "Apa saja manfaat mengonsumsi buah-buahan setiap hari?"

messages = [
    {"role": "system", "content": "Kamu adalah asisten AI yang pintar dan informatif."},
    {"role": "user", "content": prompt_text}
]

# Konversi messages ke format input model menggunakan chat template yang sudah disesuaikan
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"\nPrompt yang diformat:\n---\n{text}\n---")

# Tokenisasi input
model_inputs = tokenizer([text], return_tensors="pt").to(device)

# --- Parameter Generasi ---
max_new_tokens_to_generate = 512 # Jumlah maksimum token baru yang akan dihasilkan

print("\nMemulai pengujian performa...")

### Time To First Token (TTFT)
start_time_ttft = time.time()
with torch.no_grad():
    # Hasilkan hanya 1 token untuk mengukur TTFT
    _ = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True
    )
end_time_ttft = time.time()
ttft = end_time_ttft - start_time_ttft
print(f"**Time To First Token (TTFT): {ttft:.4f} seconds**")

### Latensi Total, Throughput, dan Perplexity
start_time_total = time.time() # Mulai waktu untuk Latensi Total

with torch.no_grad():
    generated_outputs = model.generate(
        model_inputs.input_ids,
        max_new_tokens=max_new_tokens_to_generate,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True # Sangat penting untuk menghitung perplexity
    )
end_time_total = time.time()
total_latency = end_time_total - start_time_total

# Ekstrak token yang dihasilkan (tanpa prompt input)
generated_token_ids = generated_outputs.sequences[:, model_inputs.input_ids.shape[1]:]
num_generated_tokens = generated_token_ids.shape[1]

# Menghitung Throughput (Tokens per Second)
if num_generated_tokens > 0:
    throughput = num_generated_tokens / total_latency
else:
    throughput = 0.0
print(f"**Total Latency: {total_latency:.4f} seconds**")
print(f"**Tokens Generated: {num_generated_tokens}**")
print(f"**Throughput (Tokens/second): {throughput:.2f} TPS**")

# --- Menghitung Perplexity ---
total_neg_log_likelihood = 0.0

if generated_outputs.scores:
    logits = torch.stack(generated_outputs.scores, dim=1) # Shape: (batch_size, num_generated_tokens, vocab_size)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Asumsi batch_size = 1
    for j in range(num_generated_tokens):
        token_id = generated_token_ids[0, j].item()

        # Abaikan special token (seperti EOS dan PAD) untuk perhitungan perplexity
        if token_id not in [tokenizer.eos_token_id, tokenizer.pad_token_id]:
            total_neg_log_likelihood += -log_probs[0, j, token_id].item()

    if num_generated_tokens > 0:
        average_neg_log_likelihood = total_neg_log_likelihood / num_generated_tokens
        perplexity = math.exp(average_neg_log_likelihood)
        print(f"**Perplexity dari output yang dihasilkan: {perplexity:.4f}**")
    else:
        print("Tidak ada token yang relevan yang dihasilkan untuk menghitung perplexity.")
else:
    print("Scores tidak tersedia untuk menghitung perplexity. Pastikan 'output_scores=True' diaktifkan.")

### Respons Model
response = tokenizer.batch_decode(generated_token_ids, skip_special_tokens=True)[0]
print(response)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Memuat model unsloth/Phi-3.5-mini-instruct ke cuda...
==((====))==  Unsloth 2025.7.3: Fast Llama patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model berhasil dimuat.
Chat template disesuaikan untuk Phi-3.5.

Prompt yang diformat:
---
<|system|>
Kamu adalah asisten AI yang pintar dan informatif.<|end|>
<|user|>
Jelaskan menara tertinggi di dunia.<|end|>
<|assistant|>

---

Memulai pengujian performa...
**Time To First Token (TTFT): 3.3025 seconds**
**Total Latency: 20.6504 seconds**
**Tokens Generated: 512**
**Throughput (Tokens/second): 24.79 TPS**
**Perplexity dari output yang dihasilkan: 1.2631**
Menara mencolis, Menara Petronas, di Johor, Malaysia, adalah menara tertinggi di dunia. Dinasan di sebelah enam piring, menara ini menunjukkan kesatuan dan kekayaan. Menara Petronas menjadi sebuah ikon modern, menampasti pemandian dan kemajuan Asia Tenggara.

Menara Petronas menjadi sebuah pusat turis, menarik turis dari seluruh dunia. Menara ini menjadi sebuah pusat pemandian, menampasti kekayaan dan kesatuan Asia Tenggara.

Menara Petronas menjadi sebuah pusat turis, menarik turis dari seluruh dunia. Menara ini menjadi sebuah pus

### Frasa-1B

In [ ]:
import time
import torch
import math
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# --- Konfigurasi Model dan Tokenizer ---
model_name = "nxvay/Frasa-7B-v0.2" # Model Llama 3.2 1B Instruct
max_seq_length = 8192
load_in_4bit = True
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model {model_name} to {device}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit
)
print("Model loaded successfully.")

# Ensure the model is set for inference
FastLanguageModel.for_inference(model)

# --- Adjusting Chat Template for Llama 3.x ---
# Llama 3 and its variants (3.1, 3.2) use a specific chat template.
# Unsloth provides "llama-3.1" which is suitable.
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1", # Using the specific Llama 3.1 chat template
    # Mapping can be useful if your message format is different from standard
    # {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
)
print("Chat template adjusted for Llama 3.1.")

# --- Prompt for Testing ---
prompt_text = "Nafion ditemukan pada akhir tahun berapa dan oleh siapa"
messages = [
    {"role": "system", "content": "Kamu adalah asisten AI yang pintar."},
    {"role": "user", "content": prompt_text}
]

# Convert messages to model input format using the adjusted chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"\nFormatted Prompt:\n---\n{text}\n---")

# Tokenize input
model_inputs = tokenizer([text], return_tensors="pt").to(device)

# --- Generation Parameters ---
max_new_tokens_to_generate = 512 # Maximum number of new tokens to generate

print("\nStarting performance evaluation...")

### Time To First Token (TTFT)
start_time_ttft = time.time()
with torch.no_grad():
    # Generate only 1 token to measure TTFT
    _ = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True
    )
end_time_ttft = time.time()
ttft = end_time_ttft - start_time_ttft
print(f"**Time To First Token (TTFT): {ttft:.4f} seconds**")

### Total Latency, Throughput, and Perplexity
start_time_total = time.time() # Start time for Total Latency

with torch.no_grad():
    generated_outputs = model.generate(
        model_inputs.input_ids,
        max_new_tokens=max_new_tokens_to_generate,
        use_cache=True,
        return_dict_in_generate=True,
        output_scores=True # Crucial for calculating perplexity
    )
end_time_total = time.time()
total_latency = end_time_total - start_time_total

# Extract generated tokens (without the input prompt)
generated_token_ids = generated_outputs.sequences[:, model_inputs.input_ids.shape[1]:]
num_generated_tokens = generated_token_ids.shape[1]

# Calculate Throughput (Tokens per Second)
if num_generated_tokens > 0:
    throughput = num_generated_tokens / total_latency
else:
    throughput = 0.0
print(f"**Total Latency: {total_latency:.4f} seconds**")
print(f"**Tokens Generated: {num_generated_tokens}**")
print(f"**Throughput (Tokens/second): {throughput:.2f} TPS**")

# --- Calculate Perplexity ---
total_neg_log_likelihood = 0.0

if generated_outputs.scores:
    logits = torch.stack(generated_outputs.scores, dim=1) # Shape: (batch_size, num_generated_tokens, vocab_size)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    # Assuming batch_size = 1
    for j in range(num_generated_tokens):
        token_id = generated_token_ids[0, j].item()

        # Exclude special tokens (like EOS and PAD) from perplexity calculation
        if token_id not in [tokenizer.eos_token_id, tokenizer.pad_token_id]:
            total_neg_log_likelihood += -log_probs[0, j, token_id].item()

    if num_generated_tokens > 0:
        average_neg_log_likelihood = total_neg_log_likelihood / num_generated_tokens
        perplexity = math.exp(average_neg_log_likelihood)
        print(f"**Perplexity of generated output: {perplexity:.4f}**")
    else:
        print("No relevant tokens generated to calculate perplexity.")
else:
    print("Scores not available to calculate perplexity. Ensure 'output_scores=True' is enabled.")

### Model Response
response = tokenizer.batch_decode(generated_token_ids, skip_special_tokens=True)[0]
print(response)

Loading model nxvay/Frasa-7B-v0.2 to cuda...
==((====))==  Unsloth 2025.7.3: Fast Llama patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.7.3 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Model loaded successfully.
Chat template adjusted for Llama 3.1.

Formatted Prompt:
---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Kamu adalah asisten AI yang pintar.<|eot_id|><|start_header_id|>user<|end_header_id|>

Nafion ditemukan pada akhir tahun berapa dan oleh siapa<|eot_id|><|start_header_id|>assistant<|end_header_id|>


---

Starting performance evaluation...
**Time To First Token (TTFT): 0.4153 seconds**
**Total Latency: 0.6771 seconds**
**Tokens Generated: 20**
**Throughput (Tokens/second): 29.54 TPS**
**Perplexity of generated output: 2.1157**
Nafion adalah sebuah produk sintetik yang terbentuk pada tahun 1966.
